In [1]:
description = """
This notebook improves upon z_dev7.ipynb
Target: Calculate OOD scores of steps w.r.t their agents
"""

In [96]:
from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path

import torch
import numpy as np
from tqdm import tqdm

try:
    from safetensors.torch import save_file
    from safetensors import safe_open
except ImportError:
    print("Please install safetensors: pip install safetensors")

In [5]:
@dataclass
class StepIndex:
    """Row-level metadata for one entry in the stacked gradient matrix."""
    row:        int     # row index in G
    traj_idx:   int     # 1-based index into the loaded data list
    step_idx:   int     # step index within the trajectory
    role:       str     # e.g. "WebSurfer", "Orchestrator (thought)"
    is_mistake: bool    # whether this is the gold mistake step


@dataclass
class GradientStore:
    """
    Stores gradients for multiple weight names across the same set of steps.
    """
    # Maps: weight_name -> (T, d) matrix
    Gs: dict[str, torch.Tensor] 
    
    # Metadata is shared across all layers because step indices are identical
    index:       list[StepIndex]
    lookup:      dict[tuple[int, int], int]
    traj_meta:   dict[dict]
    traj_ranges: list[tuple[int, int]]
    device:      torch.device

    def __getitem__(self, weight_name: str) -> torch.Tensor:
        return self.Gs[weight_name]

    @property
    def layer_names(self) -> list[str]:
        return list(self.Gs.keys())


def get_all_weight_names(fp: Path):
    with safe_open(fp, framework="pt") as f:
        return sorted({k.split(".", 1)[1] for k in f.keys()})
        
def load_and_stack(
    model: str,
    subset: str,
    weight_names: list[str],  # List of layers to load, e.g. ["down/31", "up/31"]
    data_dir: Path,
    device: torch.device,
    grad_dir: Path,
):
    input_dir = grad_dir / model / subset
    files = sorted(input_dir.glob("*.safetensors"), key=lambda x: int(x.stem))
    
    # Initialize containers for each requested layer
    if weight_names == "all":
        weight_names = get_all_weight_names(files[0])
        
    layer_collections = {name: [] for name in weight_names}
    index: list[StepIndex] = []
    lookup: dict[tuple[int, int], int] = {}
    traj_meta: dict[dict] = {}
    traj_ranges: list[tuple[int, int]] = []

    row = 0
    for file_idx, fp in enumerate(tqdm(files, desc="Loading Multi-Layers")):
        traj_idx = int(fp.stem) # /path/to/1.safetensors -> 1
        
        with safe_open(fp, framework="pt", device="cpu") as f:
            # Load metadata
            header = f.metadata()
            metadata = json.loads(header.get("payload_metadata", "{}"))
            mistake_step = int(metadata.get("mistake_step", -1))
            traj_meta[traj_idx] = metadata
            
            # Use the first requested layer to determine step indices 
            # (assuming all layers exist for all steps)
            first_layer = weight_names[0]
            step_keys = [k for k in f.keys() if k.endswith(f".{first_layer}")]
            step_indices = sorted([int(k.split(".")[0]) for k in step_keys])
            
            # Load matching JSON for roles
            with open(data_dir / fp.with_suffix(".json").name) as jf:
                traj_data = json.load(jf)
                history = traj_data['history']
                
            start_row = row
            for step_idx in step_indices:
                # 1. Collect tensors for EVERY requested layer at this step
                for name in weight_names:
                    key = f"{step_idx}.{name}"
                    layer_collections[name].append(f.get_tensor(key))
                
                # 2. Record index (only once per step)
                index.append(StepIndex(
                    row=row, traj_idx=traj_idx, step_idx=step_idx,
                    role=history[step_idx]["role"],
                    is_mistake=(step_idx == mistake_step),
                ))
                lookup[(traj_idx, step_idx)] = row
                row += 1

            traj_ranges.append((start_row, row))

    # Convert lists to stacked matrices and move to device
    Gs = {
        name: torch.stack(tensors).to(device) 
        for name, tensors in layer_collections.items()
    }

    return GradientStore(
        Gs=Gs, index=index, lookup=lookup,
        traj_meta=traj_meta, traj_ranges=traj_ranges,
        device=device,
    )

In [6]:
device        = torch.device("cuda:1")
grad_dir      = Path("outputs/grads")
model_name    = "llama-3.1-8b"
subset        = "hand-crafted"
# subset        = "algorithm-generated"
data_dir  = Path("ww") / subset
weight_name   = "all"

print(f"Subset:       {subset}")
print(f"Model:        {model_name}")
print(f"Device:       {device}")
print(f"Config:       {weight_name}")

store = load_and_stack(
    model=model_name,
    subset=subset,
    weight_names=weight_name,
    data_dir=data_dir,
    device=device,
    grad_dir=grad_dir
) 

Subset:       hand-crafted
Model:        llama-3.1-8b
Device:       cuda:1
Config:       all


Loading Multi-Layers:   0%|          | 0/58 [00:00<?, ?it/s]

Loading Multi-Layers: 100%|██████████| 58/58 [00:43<00:00,  1.33it/s]


In [9]:
store.Gs['down/0'].shape

torch.Size([2935, 4096])

In [10]:
import pandas as pd
import numpy as np
from typing import Callable
from scipy.stats import rankdata
from concurrent.futures import ThreadPoolExecutor
import torch

def standardize_role(role: str) -> str:
    if "orchestrator" in role.lower(): return "Orchestrator"
    else: return role

def compute_metrics(
    scores: np.ndarray,
    store: GradientStore,
    mistake_indices: list[int | None],  # absolute step_idx in history
    mistake_roles: list[str | None],
    ks: list[int],
    direction: str,
) -> dict:
    ascending    = (direction == "asc")
    total_trajs  = len(store.traj_ranges)
    step_hits    = {k: 0 for k in ks}
    agent_hits   = {k: 0 for k in ks}

    for (start, end), mistake_step, mistake_role in zip(
        store.traj_ranges, mistake_indices, mistake_roles
    ):
        if mistake_step is None:
            continue

        # Pair each entry with its score, then rank by score
        traj_entries = store.index[start:end]
        traj_scores  = scores[start:end]
        step_scores  = [(entry.step_idx, entry.role, score) 
                        for entry, score in zip(traj_entries, traj_scores)]
        step_scores.sort(key=lambda x: x[2], reverse=not ascending)

        ranked_steps  = [step_idx for step_idx, _, _ in step_scores]
        ranked_roles  = [standardize_role(role) for _, role, _ in step_scores]
        mistake_rank  = ranked_steps.index(mistake_step) + 1  # 1-based ranking.

        for k in ks:
            if mistake_rank <= k:
                step_hits[k] += 1
            if mistake_role in ranked_roles[:k]:
                agent_hits[k] += 1

    return {
        **{f"step@{k}_{direction}":  step_hits[k]  / total_trajs for k in ks},
        **{f"agent@{k}_{direction}": agent_hits[k] / total_trajs for k in ks},
    }

def evaluate_weights(
    store: GradientStore,
    scoring_fn: Callable[[torch.Tensor], torch.Tensor],
    ks: list[int] = [1, 3, 5],
) -> pd.DataFrame:

    # --- Phase 1: Compute scores for all weights ---
    all_scores: dict[str, np.ndarray] = {}
    for weight_name, G in tqdm(store.Gs.items(), desc="Scoring"):
        all_scores[weight_name] = scoring_fn(G).cpu().numpy()

    # --- Precompute trajectory metadata ---
    mistake_indices: list[int | None] = []
    mistake_roles:   list[str | None] = []

    for start, end in store.traj_ranges:
        traj_index  = store.index[start:end]
        mistake_entry = next((e for e in traj_index if e.is_mistake), None)
        mistake_role = store.traj_meta[mistake_entry.traj_idx]['mistake_agent']
        mistake_idx = mistake_entry.step_idx

        mistake_roles.append(mistake_role)
        mistake_indices.append(mistake_idx)
        # mistake_roles.append(mistake_entry.role if mistake_entry else None)

    # --- Phase 2: Evaluate predictions (parallelized over weights) ---
    results = []
    for weight_name, scores in tqdm(all_scores.items(), desc="Predicting"):
        row = {"weight": weight_name}
        for direction in ["asc", "desc"]:
            row |= compute_metrics(
                scores, 
                store, 
                mistake_indices, 
                mistake_roles, 
                ks, 
                direction
            )
        results.append(row)

    df = pd.DataFrame(results)
    df = df.sort_values("step@1_asc", ascending=False).reset_index(drop=True)
    return df

In [ ]:

# len([x for x in ROLES if "Web" in x])

In [31]:
store.Gs['down/0'][[0, 12]]

tensor([[ 0.0463, -0.0073,  0.0154,  ..., -0.0189, -0.0035, -0.0205],
        [ 0.0201,  0.0490, -0.0327,  ..., -0.0551, -0.0005,  0.1053]],
       device='cuda:1', dtype=torch.float16)

In [46]:
def group_role(role: str) -> str:
    if role.startswith("Orchestrator (->"): return "Orchestrator (-> Agent)"
    else: return role

G = store.Gs['down/0']
index = store.index
ROLES = sorted(set(group_role(idx.role) for idx in index))

role_Gs = dict()
for role in set(ROLES):
    role_mask = torch.tensor(
        [group_role(idx.role) == role for idx in index], 
        device=G.device
    )
    role_G = G[role_mask]
    role_Gs[role] = role_G

In [48]:
ROLES

['Assistant',
 'ComputerTerminal',
 'FileSurfer',
 'Orchestrator (-> Agent)',
 'Orchestrator (termination condition)',
 'Orchestrator (thought)',
 'WebSurfer']

In [63]:
role_idxs = [idx for idx in store.index if 
             group_role(idx.role) == "ComputerTerminal"
             and idx.is_mistake
             ]
role_idxs

[]

In [64]:
# NOTE: ComputerTerminal and Orchestrator (termnination condition) steps are never error steps.

In [127]:
import torch
from typing import Callable


# ═══════════════════════════════════════════════════════════════════
# Family 1: Central-tendency references
#   Score = distance (or similarity) to a single summary vector.
#   Higher score = more anomalous, EXCEPT cosine_similarity which
#   returns similarity (higher = more normal).
# ═══════════════════════════════════════════════════════════════════

def cosine_similarity(G: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Cosine similarity of each row to the mean row of G.

    NOTE: Higher = more *normal* (opposite of the other scorers here).
    Negate the output if you want a unified "higher = more OOD" convention.
    """
    reference_raw = G.mean(dim=0)
    ref_norm = torch.norm(reference_raw) + eps
    reference_normalized = reference_raw / ref_norm

    row_norms = torch.norm(G, dim=1, keepdim=True) + eps
    normalized_G = G / row_norms

    return torch.matmul(normalized_G, reference_normalized)


def mean_distance(G: torch.Tensor) -> torch.Tensor:
    """L2 distance of each row to the arithmetic mean of G.

    Baseline non-robust reference. Breakdown point 1/N.
    """
    G_f = G.float()
    mu = G_f.mean(dim=0)
    return torch.norm(G_f - mu, dim=1).to(G.dtype)


def coordinate_median(G: torch.Tensor) -> torch.Tensor:
    """L2 distance of each row to the coordinate-wise median of G.

    Cheap, but not rotation-invariant. Useful as a sanity baseline.
    """
    G_f = G.float()
    mu = G_f.median(dim=0).values
    return torch.norm(G_f - mu, dim=1).to(G.dtype)


def geometric_median(
    G: torch.Tensor,
    eps: float = 1e-8,
    max_iter: int = 100,
    tol: float = 1e-6,
) -> torch.Tensor:
    """L2 distance of each row to the geometric median (Weiszfeld iteration).

    Rotation-invariant, breakdown ≈ 50%. Drop-in robust replacement for
    the arithmetic mean.
    """
    G_f = G.float()
    mu = G_f.mean(dim=0)  # warm-start

    for _ in range(max_iter):
        d = torch.norm(G_f - mu, dim=1) + eps
        w = 1.0 / d
        mu_new = (w.unsqueeze(1) * G_f).sum(dim=0) / w.sum()
        if torch.norm(mu_new - mu) < tol:
            mu = mu_new
            break
        mu = mu_new

    return torch.norm(G_f - mu, dim=1).to(G.dtype)


# ═══════════════════════════════════════════════════════════════════
# Family 2: Spectral / subspace-based
#   Score = projection onto (or residual from) an SVD subspace.
# ═══════════════════════════════════════════════════════════════════

def _run_svd(G: torch.Tensor, c: int) -> torch.Tensor:
    """Top-c right singular vectors of G, shape (d, c)."""
    _, _, V = torch.svd_lowrank(G, q=c, niter=10)
    return V


def sal_projection(
    G: torch.Tensor,
    c: int = 1,
    centered: bool = True,
) -> torch.Tensor:
    """Mean squared projection onto the top-c SVD directions.

        τ_i = (1/c) * Σ_{j=1}^{c} ⟨g̃_i, v_j⟩²

    SAL's interpretation (Du et al. 2024): the top singular direction
    aligns with the *outlier* direction, so a large projection flags
    likely OOD rows.
    """
    G_f = G.float()
    if centered:
        G_f = G_f - G_f.mean(dim=0)
    V = _run_svd(G_f, c)                              # (d, c)
    return (G_f @ V).square().mean(dim=1).to(G.dtype)


def reconstruction_svd(
    G: torch.Tensor,
    c: int = 5,
    centered: bool = True,
) -> torch.Tensor:
    """Residual L2 norm after projecting onto the top-c SVD subspace.

    Complement of `sal_projection`: the top-c subspace captures the
    dominant structure of G, and rows off this subspace have large
    residuals. Interpretation flips depending on whether the majority
    or the outliers dominate the top directions.
    """
    G_f = G.float()
    if centered:
        G_c = G_f - G_f.mean(dim=0)
    else:
        G_c = G_f

    V = _run_svd(G_c, c)                              # (d, c)
    proj  = G_c @ V                                   # (N, c)
    G_rec = proj @ V.T                                # (N, d)
    return torch.norm(G_c - G_rec, dim=1).to(G.dtype)


# ═══════════════════════════════════════════════════════════════════
# Family 3: Neighborhood-based
#   Score = isolation from nearby rows (no global reference).
# ═══════════════════════════════════════════════════════════════════

def knn_distance(
    G: torch.Tensor,
    k: int = 5,
    normalize: bool = True,
) -> torch.Tensor:
    """Distance to the k-th nearest neighbor within G (excluding self).

    Follows Sun et al. 2022. With `normalize=True`, uses unit-normalized
    rows so L2 distance is monotone in cosine distance (the variant in
    the original paper). Isolated points have large k-NN distances.
    """
    G_f = G.float()
    N = G_f.shape[0]
    k_eff = max(1, min(k, N - 1))

    if normalize:
        G_f = G_f / (torch.norm(G_f, dim=1, keepdim=True) + 1e-8)

    D = torch.cdist(G_f, G_f, p=2)                    # (N, N)
    D.fill_diagonal_(float("inf"))
    kth, _ = D.kthvalue(k_eff, dim=1)                 # (N,)
    return kth.to(G.dtype)


# ═══════════════════════════════════════════════════════════════════
# Role-based orchestration (unchanged)
# ═══════════════════════════════════════════════════════════════════

def group_role(role: str) -> str:
    if role.startswith("Orchestrator (->"):
        return "Orchestrator (-> Agent)"
    else:
        return role


def split_by_role(G: torch.Tensor, index: list) -> dict:
    ROLES = set(group_role(idx.role) for idx in index)
    role_Gs = dict()
    for role in ROLES:
        role_idxs = [idx.row for idx in index if group_role(idx.role) == role]
        role_mask = torch.tensor(
            [group_role(idx.role) == role for idx in index],
            device=G.device,
        )
        role_G = G[role_mask]
        role_Gs[role] = {"idx": role_idxs, "G": role_G}
    return role_Gs


def compute_split_scores(
    G: torch.Tensor,
    index: list,
    scoring: Callable,
) -> torch.Tensor:
    role_Gs = split_by_role(G, index)
    scores = torch.zeros(G.shape[0], device=G.device, dtype=G.dtype)
    for role_name, role_data in role_Gs.items():
        role_G = role_data["G"]
        role_scores = scoring(role_G)
        scores[role_data["idx"]] = role_scores
    return scores

def compute_scores(
    G: torch.Tensor,
    index: list,
    scoring: Callable,
) -> torch.Tensor:
    scores = scoring(G)
    return scores


def make_scoring_fn(index, scoring, name: str = "scoring_by_role"):
    def scoring_fn(G: torch.Tensor) -> torch.Tensor:
        return compute_scores(G, index, scoring)
    scoring_fn.__name__ = name
    return scoring_fn


# ═══════════════════════════════════════════════════════════════════
# Factories — one per scorer, matching your `make_sal_scoring` pattern
# ═══════════════════════════════════════════════════════════════════

# Family 1: central-tendency
def make_cosine_scoring():
    def scoring(G): return cosine_similarity(G)
    return scoring

def make_mean_distance_scoring():
    def scoring(G): return mean_distance(G)
    return scoring

def make_coordinate_median_scoring():
    def scoring(G): return coordinate_median(G)
    return scoring

def make_geometric_median_scoring(max_iter: int = 100, tol: float = 1e-6):
    def scoring(G): return geometric_median(G, max_iter=max_iter, tol=tol)
    return scoring

# Family 2: spectral
def make_sal_scoring(c: int = 1, centered: bool = True):
    def scoring(G): return sal_projection(G, c=c, centered=centered)
    return scoring

def make_reconstruction_scoring(c: int = 5, centered: bool = True):
    def scoring(G): return reconstruction_svd(G, c=c, centered=centered)
    return scoring

# Family 3: neighborhood
def make_knn_scoring(k: int = 5, normalize: bool = True):
    def scoring(G): return knn_distance(G, k=k, normalize=normalize)
    return scoring

In [128]:
# ── Full hyperparameter sweep ──────────────────────────────────────────────────
from itertools import product as iproduct

all_results: dict[str, pd.DataFrame] = {}

# ── Family 1: Central-tendency (no hyperparams) ────────────────────────────────
for name, factory in [
    ("cosine",       make_cosine_scoring),
    ("mean_dist",    make_mean_distance_scoring),
    ("coord_median", make_coordinate_median_scoring),
]:
    key = name
    all_results[key] = evaluate_weights(
        store,
        scoring_fn=make_scoring_fn(store.index, factory(), name=name),
    )

# geom_median: sweep max_iter robustness (tol is less interesting)
for max_iter in [50, 100, 200]:
    key = f"geom_median_iter{max_iter}"
    all_results[key] = evaluate_weights(
        store,
        scoring_fn=make_scoring_fn(
            store.index,
            make_geometric_median_scoring(max_iter=max_iter),
            name=key,
        ),
    )

# ── Family 2: Spectral ─────────────────────────────────────────────────────────
# sal_projection: sweep c ∈ {1..5} × centered ∈ {True, False}
for c, centered in iproduct(range(1, 6), [True, False]):
    key = f"sal_c{c}_{'cen' if centered else 'raw'}"
    all_results[key] = evaluate_weights(
        store,
        scoring_fn=make_scoring_fn(
            store.index,
            make_sal_scoring(c=c, centered=centered),
            name=key,
        ),
    )

# reconstruction_svd: sweep c ∈ {1..8} × centered ∈ {True, False}
for c, centered in iproduct(range(1, 9), [True, False]):
    key = f"recon_c{c}_{'cen' if centered else 'raw'}"
    all_results[key] = evaluate_weights(
        store,
        scoring_fn=make_scoring_fn(
            store.index,
            make_reconstruction_scoring(c=c, centered=centered),
            name=key,
        ),
    )

# ── Family 3: kNN ──────────────────────────────────────────────────────────────
for k, normalize in iproduct([1, 3, 5, 10, 20], [True, False]):
    key = f"knn_k{k}_{'norm' if normalize else 'raw'}"
    all_results[key] = evaluate_weights(
        store,
        scoring_fn=make_scoring_fn(
            store.index,
            make_knn_scoring(k=k, normalize=normalize),
            name=key,
        ),
    )

# ── Find best configs ──────────────────────────────────────────────────────────
summary_rows = []
for config_name, df in all_results.items():
    best_weight = df.iloc[0]["weight"]                   # already sorted by step@1_asc
    summary_rows.append({
        "config":        config_name,
        "best_weight":   best_weight,
        "step@1_asc":    df.iloc[0]["step@1_asc"],
        "step@1_desc":   df["step@1_desc"].max(),
        "best_w_desc":   df.loc[df["step@1_desc"].idxmax(), "weight"],
    })

summary = pd.DataFrame(summary_rows)

best_asc  = summary.loc[summary["step@1_asc"].idxmax()]
best_desc = summary.loc[summary["step@1_desc"].idxmax()]

print("━━ Best config for step@1_asc ━━")
print(best_asc.to_string())
print("\n━━ Best config for step@1_desc ━━")
print(best_desc.to_string())

summary.sort_values("step@1_asc", ascending=False).head(10)

Scoring:   0%|          | 0/291 [00:00<?, ?it/s]

Predicting: 100%|██████████| 291/291 [00:01<00:00, 211.25it/s]

━━ Best config for step@1_asc ━━
config           sal_c1_raw
best_weight    post_norm/10
step@1_asc         0.293103
step@1_desc        0.051724
best_w_desc      in_norm/21

━━ Best config for step@1_desc ━━
config         knn_k10_norm
best_weight      in_norm/28
step@1_asc         0.172414
step@1_desc        0.224138
best_w_desc            o/12


,config,best_weight,step@1_asc,step@1_desc,best_w_desc
7,sal_c1_raw,post_norm/10,0.293103,0.051724,in_norm/21
29,recon_c7_raw,o/15,0.293103,0.086207,up/26
31,recon_c8_raw,o/15,0.293103,0.086207,up/26
27,recon_c6_raw,o/15,0.293103,0.103448,gate/25
5,geom_median_iter200,down/6,0.275862,0.034483,k/31
25,recon_c5_raw,o/15,0.275862,0.103448,gate/25
15,sal_c5_raw,up/18,0.275862,0.034483,o/5
3,geom_median_iter50,down/6,0.275862,0.034483,k/31
4,geom_median_iter100,down/6,0.275862,0.034483,k/31
2,coord_median,down/6,0.275862,0.034483,k/31


In [158]:
all_results['sal_c7_raw'].head(10)

KeyError: 'sal_c7_raw'

In [90]:
df = evaluate_weights(store, scoring_fn=make_scoring_fn(store.index, scoring=cosine_similarity))

Predicting: 100%|██████████| 291/291 [00:01<00:00, 204.63it/s]


In [91]:
df

,weight,step@1_asc,step@3_asc,step@5_asc,agent@1_asc,agent@3_asc,agent@5_asc,step@1_desc,step@3_desc,step@5_desc,agent@1_desc,agent@3_desc,agent@5_desc
0,gate/29,0.206897,0.396552,0.448276,0.586207,0.810345,0.896552,0.017241,0.051724,0.103448,0.310345,0.362069,0.379310
1,in_norm/27,0.172414,0.344828,0.431034,0.551724,0.844828,0.896552,0.000000,0.068966,0.155172,0.310345,0.327586,0.396552
2,in_norm/19,0.172414,0.258621,0.293103,0.517241,0.758621,0.862069,0.000000,0.051724,0.120690,0.310345,0.344828,0.362069
3,gate/17,0.172414,0.258621,0.327586,0.482759,0.741379,0.844828,0.000000,0.086207,0.137931,0.310345,0.344828,0.362069
4,o/17,0.172414,0.293103,0.362069,0.551724,0.810345,0.913793,0.000000,0.086207,0.103448,0.310345,0.344828,0.362069
...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,post_norm/23,0.000000,0.068966,0.137931,0.310345,0.344828,0.448276,0.017241,0.086207,0.103448,0.310345,0.344828,0.379310
287,gate/23,0.000000,0.086207,0.155172,0.293103,0.362069,0.500000,0.000000,0.086207,0.120690,0.310345,0.344828,0.396552
288,gate/10,0.000000,0.103448,0.172414,0.465517,0.655172,0.844828,0.017241,0.086207,0.120690,0.310345,0.344828,0.379310
289,gate/26,0.000000,0.103448,0.189655,0.310345,0.362069,0.448276,0.017241,0.103448,0.120690,0.310345,0.344828,0.379310


In [76]:
compute_scores(store['down/0'], store.index, cosine_similarity)

tensor([0.1959, 0.1786, 0.8975,  ..., 0.1302, 0.1820, 0.5288], device='cuda:1',
       dtype=torch.float16)

In [ ]:
df = evaluate_weights(store, scoring_fn=compute_scores(c=1, centered=False))

In [ ]:
"""sal_scoring.py — SAL scoring functions compatible with evaluate_weights."""

import torch


def _run_svd(G: torch.Tensor, c: int) -> torch.Tensor:
    """Return top-c right singular vectors, shape (d, c)."""
    _, _, V = torch.svd_lowrank(G, q=c, niter=10)
    return V


def _sal(G: torch.Tensor, c: int, centered: bool) -> torch.Tensor:
    """Score each row of G by mean squared projection onto top-c SVD vectors.

    τ_i = (1/c) * Σ_{j=1}^{c} ⟨g̃_i, v_j⟩²
    """
    G_f = G.float()
    if centered:
        G_f = G_f - G_f.mean(dim=0)
    V = _run_svd(G_f, c)          # (d, c)
    return (G_f @ V).square().mean(dim=1)   # (T,)


def make_sal_scoring_fn(c: int = 1, centered: bool = True):
    """Factory: returns a scoring_fn(G: Tensor) -> (T,) for evaluate_weights.

    Args:
        c:        number of singular vectors (1–5).
        centered: if True, subtract the mean gradient (SAL w/ ref);
                  otherwise use raw gradients (SAL w/o ref).
    """
    def scoring_fn(G: torch.Tensor) -> torch.Tensor:
        return _sal(G, c=c, centered=centered)
    scoring_fn.__name__ = f"sal_{'wref' if centered else 'noref'}_c{c}"
    return scoring_fn


# All 10 pre-built variants (sal_wref_c1..5, sal_noref_c1..5)
SAL_SCORING_FNS: dict[str, callable] = {
    f"sal_{'wref' if centered else 'noref'}_c{c}": make_sal_scoring_fn(c, centered)
    for centered in (True, False)
    for c in range(1, 6)
}

In [124]:
from grads.sal_scoring import make_sal_scoring_fn, SAL_SCORING_FNS

# Single variant
df = evaluate_weights(store, scoring_fn=make_sal_scoring_fn(c=1, centered=False))

Predicting: 100%|██████████| 291/291 [00:01<00:00, 205.85it/s]


In [125]:
df

,weight,step@1_asc,step@3_asc,step@5_asc,agent@1_asc,agent@3_asc,agent@5_asc,step@1_desc,step@3_desc,step@5_desc,agent@1_desc,agent@3_desc,agent@5_desc
0,post_norm/10,0.293103,0.362069,0.465517,0.620690,0.741379,0.775862,0.000000,0.017241,0.103448,0.310345,0.310345,0.344828
1,down/6,0.258621,0.362069,0.465517,0.534483,0.741379,0.810345,0.000000,0.034483,0.103448,0.310345,0.310345,0.344828
2,down/2,0.258621,0.379310,0.413793,0.655172,0.793103,0.879310,0.000000,0.034483,0.103448,0.310345,0.310345,0.344828
3,v/8,0.224138,0.362069,0.396552,0.620690,0.844828,0.896552,0.000000,0.051724,0.103448,0.310345,0.310345,0.344828
4,post_norm/15,0.206897,0.396552,0.482759,0.586207,0.793103,0.862069,0.000000,0.017241,0.103448,0.310345,0.310345,0.344828
...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,in_norm/19,0.034483,0.258621,0.344828,0.327586,0.758621,0.879310,0.000000,0.051724,0.103448,0.310345,0.327586,0.362069
287,q/22,0.017241,0.241379,0.413793,0.413793,0.810345,0.931034,0.017241,0.086207,0.103448,0.310345,0.344828,0.396552
288,in_norm/28,0.017241,0.103448,0.258621,0.275862,0.620690,0.879310,0.017241,0.051724,0.137931,0.327586,0.362069,0.448276
289,k/27,0.017241,0.172414,0.310345,0.379310,0.793103,0.896552,0.000000,0.068966,0.103448,0.310345,0.344828,0.362069
